In [1]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

## Model Categories

### Ultra-Light (<1B)
- TinyLlama-1.1B
- Qwen2-0.5B
- Phi-2

### Small (1B–3B)
- Gemma-2B
- StableLM-3B
- Falcon-3B

### Medium (3B–7B)
- Mistral-7B-Instruct
- LLaMA-3-8B
- Gemma-7B

### Large (7B–13B)
- LLaMA-2-13B
- Nous-Hermes-13B
- RedPajama-7B

### Extra Large (13B+)
- Mixtral-8x7B
- Qwen-14B
- LLaMA-3-70B

In [ ]:
models = {
    "Ultra-Light": [
        "Qwen/Qwen2.5-0.5B-Instruct",
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "microsoft/phi-2"
    ],

    "Small": [
        "google/gemma-2b",
        "ibm-granite/granite-3.1-2b-instruct",
        "stabilityai/stablelm-3b-4e1t"
    ],

    "Medium": [
        "mistralai/Mistral-7B-Instruct-v0.1",
        "google/gemma-7b",
        "tiiuae/falcon-7b"
    ]

     "Large": [
        "meta-llama/Llama-2-13b-hf",
        "NousResearch/Nous-Hermes-13b",
        "mistralai/Mixtral-8x7B"
    ]

    # X- Large
    #Due to hardware limitations, models above 13B were evaluated based on leaderboard documentation rather than full local execution.
}

In [2]:
def load_model(model_name):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    return tokenizer, model

In [ ]:
test_prompt = "Extract structured JSON from: 'Math tutoring for $20/hour."

In [ ]:
def generate_response(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    end = time.time()

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("Response:\n", response)
    print("Time:", round(end - start, 2), "seconds")

In [ ]:
for category, model_list in models.items():
    for model_name in model_list:
        try:
            tokenizer, model = load_model(model_name)
            generate_response(tokenizer, model, test_prompt)
        except Exception as e:
            print(f"Failed to load {model_name}")
            print(e)